# Derin Öğrenme ve CNN Yapısının İncelenmesi (Fashion-MNIST)

**Ad Soyad:** [Adınız Soyadınız]  
**Öğrenci No:** [Numaranız]  
**Ders:** Derin Öğrenme / Yapay Zeka  

Bu çalışmada Evrişimsel Sinir Ağlarının (CNN) temel mantığını, katmanların matris boyutlarını nasıl değiştirdiğini inceledim ve Fashion-MNIST veri seti üzerinde basit bir CNN modeli eğittim.

## 2. Bölüm: Temel Kavramlar

### Soru 1. Derin Öğrenme ve Sinir Ağları
- **Makine Öğrenmesi ile Derin Öğrenme Farkı:** Klasik makine öğrenmesinde özellikleri (features) elle çıkarmak gerekir. Derin öğrenmede ise çok katmanlı ağlar piksellerden özellikleri otomatik öğrenir.
- **Yapay Sinir Ağları Nasıl Çalışır:** Girdiler ağırlıklarla çarpılıp bias eklenir, aktivasyondan geçirilir. Tahmin yapıldıktan sonra kayıp (loss) hesaplanır ve geriye yayılım ile ağırlıklar güncellenir.
- **Neden Birden Fazla Katman:** Tek katman sadece doğrusal ayrımları yapabilir. Birden fazla katman olunca ilk katmanlar basit çizgileri, sonraki katmanlar şekil ve nesneleri hiyerarşik olarak öğrenir.

### Soru 2. Konvolüsyon ve Filtreler
- **Konvolüsyon İşlemi:** Küçük bir filtre matrisi görüntü üzerinde kaydırılarak altındaki piksellerle çarpılıp toplanır ve özellik haritası üretilir.
- **Kernel (Filtre):** Belirli desenleri (kenar, köşe vb.) yakalamak için eğitilen küçük ağırlık matrisidir.
- **3x3 Filtre Parametresi:** Tek kanallı giriş için 3x3x1 = 9 ağırlık vardır.
- **Filtre Gezerken Ne Yapar:** Her bölgede piksellerle benzerliği ölçer; desen varsa yüksek sayı üretir.
- **Neden Birden Fazla Filtre:** Her filtre tek bir deseni yakalar. Farklı yönlerdeki çizgileri ve dokuları yakalamak için çok sayıda filtre kullanılır.

### Soru 3. Pooling (Havuzlama)
- **Max Pooling Amacı:** Boyutu küçültmek, parametre sayısını azaltmak, ezberlemeyi (overfitting) önlemek ve nesne hafif kaydığında bile yakalanmasını sağlamaktır.
- **2x2 Max Pooling:** 2x2'lik pencerelerdeki en büyük sayıyı seçer.
- **Boyut Değişimi:** Genişlik ve yükseklik yarıya iner, toplam piksel sayısı %75 azalır.

### Soru 4. Aktivasyon Fonksiyonları
- **Aktivasyon Fonksiyonlarının Amacı:** Ağa doğrusal olmama özelliği kazandırarak karmaşık desenleri öğrenebilmesini sağlamaktır.
- **ReLU:** $f(x) = \max(0, x)$. Negatifleri sıfırlar, pozitifleri aynen geçirir. Gradyan kaybolmasını önler ve çok hızlıdır.
- **Softmax:** Son katmandaki skorları toplamı 1 olan olasılıklara dönüştürür.
- **ReLU ile Softmax Farkı:** ReLU ara katmanlarda bağımsız çalışır; Softmax son katmanda tüm sınıfları birbirine bağlayarak olasılık dağılımı verir.

### Soru 5. Dense Katman
- **Dense Katman:** Tüm nöronların birbirine bağlı olduğu klasik katmandır.
- **Flatten:** Konvolüsyondan çıkan çok boyutlu tensörü tek boyutlu düz diziye çevirir (örn: 5x5x16 -> 400).
- **Neden Kullanılır:** Konvolüsyon yerel özellikleri yakalar, Dense katman ise bu özellikleri birleştirerek son sınıflandırma kararını verir.

## 3. Bölüm: Konvolüsyon İşleminin Kodlanması

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

# Rastgelelik kontrolü
tf.keras.utils.set_random_seed(42)

# 3x3 boyutunda 16 filtreli konvolüsyon katmanı
conv = tf.keras.layers.Conv2D(
    filters=16,
    kernel_size=(3, 3),
    activation='relu',
    kernel_initializer='he_normal'
)

# Örnek giriş görüntüsü (1 adet, 128x128 gri görüntü)
dummy_input = tf.random.normal((1, 128, 128, 1))

# Konvolüsyon işlemi
output = conv(dummy_input)

# Ağırlıkları al
weights, biases = conv.get_weights()

print("Giriş boyutu              :", dummy_input.shape)
print("Çıkış boyutu              :", output.shape)
print("Filtre ağırlıklarının boyutu:", weights.shape)
print("Bias boyutu               :", biases.shape)
print("\nİlk filtrenin ağırlıkları (3x3):")
print(weights[:, :, 0, 0])

### 3. Bölüm Analiz ve Filtrelerin Çizdirilmesi
- **16 Filtre:** Girişten 16 ayrı özellik haritası çıkarılır.
- **Filtre boyutu:** 3x3x1 = 9 ağırlık.
- **weights (3, 3, 1, 16):** Toplam 144 ağırlık.
- **biases (16,):** 16 filtre için 16 bias. Toplam parametre = 160.
- **128'den 126'ya iniş:** Padding 'valid' olduğu için kenarlardan 1'er piksel daralmıştır: $(128 - 3) + 1 = 126$.
- **he_normal:** Ağırlıkların ReLU ile uyumlu normal dağılımla başlatılmasıdır.

In [ ]:
# 16 Filtrenin Başlangıç Ağırlıklarını Görselleştirme
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
fig.suptitle("16 Başlangıç Konvolüsyon Filtresi (3x3)", fontsize=13, weight='bold')

for i in range(16):
    ax = axes[i // 4, i % 4]
    kernel = weights[:, :, 0, i]
    im = ax.imshow(kernel, cmap='coolwarm', interpolation='nearest')
    ax.set_title(f"Filtre #{i+1}", fontsize=9)
    ax.axis('off')
    for r in range(3):
        for c in range(3):
            ax.text(c, r, f"{kernel[r, c]:.2f}", ha="center", va="center", color="black", fontsize=7)

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/bolum3_filtreler.png", dpi=300)
plt.show()

## 4. Bölüm: Pooling İşleminin İncelenmesi

In [ ]:
# 4x4 giriş matrisi
input_data = tf.constant([
    [[1., 3., 2., 4.],
     [5., 6., 1., 2.],
     [7., 2., 8., 3.],
     [4., 1., 5., 9.]]
])

input_data = tf.reshape(input_data, (1, 4, 4, 1))
pool = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2)
output_pool = pool(input_data)

print("Giriş Matrisi (4x4):\n", input_data.numpy()[0, :, :, 0])
print("\nPooling Sonucu (2x2):\n", output_pool.numpy()[0, :, :, 0])

### Elle Hesaplama ve Karşılaştırma
- Sol-üst (1,3,5,6) -> max: **6**
- Sağ-üst (2,4,1,2) -> max: **4**
- Sol-alt (7,2,4,1) -> max: **7**
- Sağ-alt (8,3,5,9) -> max: **9**

Elle hesaplanan: `[[6, 4], [7, 9]]`. Program çıktısı ile **birebir aynıdır**.
Boyut (1, 4, 4, 1) -> (1, 2, 2, 1) yarı yarıya düşmüştür.

In [ ]:
mat_in = input_data.numpy()[0, :, :, 0]
mat_out = output_pool.numpy()[0, :, :, 0]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(mat_in, cmap='Blues', vmin=0, vmax=10)
axes[0].set_title("Giriş Matrisi (4x4)", fontsize=11, weight='bold')
for r in range(4):
    for c in range(4):
        axes[0].text(c, r, f"{mat_in[r, c]:.0f}", ha='center', va='center', color='black' if mat_in[r,c]<7 else 'white', fontsize=11, weight='bold')
axes[0].axvline(1.5, color='red', linestyle='--', linewidth=2)
axes[0].axhline(1.5, color='red', linestyle='--', linewidth=2)

axes[1].imshow(mat_out, cmap='Greens', vmin=0, vmax=10)
axes[1].set_title("2x2 Max Pooling Sonucu", fontsize=11, weight='bold')
for r in range(2):
    for c in range(2):
        axes[1].text(c, r, f"{mat_out[r, c]:.0f}", ha='center', va='center', color='black' if mat_out[r,c]<7 else 'white', fontsize=13, weight='bold')

plt.tight_layout()
plt.savefig("figures/bolum4_maxpooling.png", dpi=300)
plt.show()

## 5. Bölüm: ReLU ve Softmax İncelenmesi

In [ ]:
# ReLU
x = tf.constant([-5., -2., -1., 0., 2., 4., 6.])
relu = tf.keras.layers.ReLU()
y_relu = relu(x)
print("Giriş       :", x.numpy())
print("ReLU Çıkışı :", y_relu.numpy())

# Softmax
logits = tf.constant([[2.0, 1.0, 0.1]])
softmax = tf.keras.layers.Softmax()
probabilities = softmax(logits)
print("\nSoftmax Sonuçları     :", probabilities.numpy())
print("Olasılıkların Toplamı :", tf.reduce_sum(probabilities).numpy())
print("En Yüksek Sınıf       :", tf.argmax(probabilities, axis=1).numpy()[0])

### 5. Bölüm Değerlendirmesi
- ReLU negatifleri sıfırladı, pozitifleri korudu. Böylece gereksiz nöronlar susturulur.
- Softmax çıktıları [0.659, 0.242, 0.099] oldu. Toplamları tam 1.0'dır. En yüksek sınıf Sınıf 0 (%65.9) olarak belirlendi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x_vals = np.linspace(-6, 6, 200)
axes[0].plot(x_vals, np.maximum(0, x_vals), color='#1f77b4', linewidth=2.5, label='f(x)=max(0, x)')
axes[0].scatter(x.numpy(), y_relu.numpy(), color='red', s=50, zorder=5, label='Örnekler')
axes[0].set_title("ReLU Fonksiyonu", fontsize=11, weight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

classes = ["Sınıf 0", "Sınıf 1", "Sınıf 2"]
axes[1].bar(classes, probabilities.numpy()[0], color=['#2ca02c', '#1f77b4', '#ff7f0e'], width=0.45)
axes[1].set_title("Softmax Olasılıkları (Toplam=1.0)", fontsize=11, weight='bold')
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("figures/bolum5_aktivasyonlar.png", dpi=300)
plt.show()

## 6. Bölüm: Fashion-MNIST ile CNN Modeli

In [ ]:
# Veriyi yükleme ve normalize etme
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# Model mimarisi
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(8, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])
model.summary()

# Derleme ve eğitim
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Eğitim grafiklerini çizdirme
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Eğitim', marker='o')
plt.plot(history.history['val_accuracy'], label='Doğrulama', marker='s')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Eğitim', marker='o')
plt.plot(history.history['val_loss'], label='Doğrulama', marker='s')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/bolum6_egitim_grafigi.png", dpi=300)
plt.show()

## 7. Bölüm: Model Mimarisinin İncelenmesi

1. **Konvolüsyon katmanı sayısı:** 2 adet.
2. **İlk katmanda neden 8 filtre:** Temel çizgileri ve kenarları yakalamak için yeterlidir, işlem yükünü düşük tutar.
3. **İkinci katmanda neden 16 filtre:** Daha derin katmanda daha karmaşık parça kombinasyonlarını yakalamak için artırılmıştır.
4. **Boyut değişimleri:** 28x28 -> 26x26 (Conv1) -> 13x13 (Pool1) -> 11x11 (Conv2) -> 5x5 (Pool2).
5. **Pooling katkısı:** Boyutu küçülterek parametre sayısını azaltır ve ezberlemeyi önler.
6. **Flatten çıktısı:** $5 \times 5 \times 16 = 400$ elemanlı vektör.
7. **Dense 64 sayısı:** Bu katmandaki nöron sayısıdır.
8. **Son katmanda 10 nöron:** Fashion-MNIST'te 10 farklı kıyafet sınıfı olduğu için.
9. **Softmax en yüksek değeri:** Modelin tahmin ettiği sınıfı ve güven yüzdesini gösterir.
10. **Başarım metriği:** Accuracy (Doğruluk) metriği.

## 8. Bölüm: Küçük Bir Deney (8 Filtre vs 16 Filtre)

In [ ]:
import time

def train_exp_model(first_filters):
    tf.keras.utils.set_random_seed(42)
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(first_filters, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(16, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    t0 = time.time()
    h = m.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1, verbose=0)
    t_diff = time.time() - t0
    
    l, acc = m.evaluate(x_test, y_test, verbose=0)
    return h.history['accuracy'][-1], acc, t_diff, h

print("Deney 1 (8 filtre) eğitiliyor...")
tr1, te1, t1, h1 = train_exp_model(8)

print("Deney 2 (16 filtre) eğitiliyor...")
tr2, te2, t2, h2 = train_exp_model(16)

print(f"Deney 1 (8 Filtre)  : Train: %{tr1*100:.2f} | Test: %{te1*100:.2f} | Süre: {t1:.1f}s")
print(f"Deney 2 (16 Filtre) : Train: %{tr2*100:.2f} | Test: %{te2*100:.2f} | Süre: {t2:.1f}s")

### Deney Tablosu ve Yorum

| Ölçüt | Deney 1 | Deney 2 |
| :--- | :---: | :---: |
| **Filtre sayısı** | **8** | **16** |
| **Eğitim doğruluğu** | %88.52 | %89.41 |
| **Test doğruluğu** | %87.35 | %88.24 |
| **Eğitim süresi** | ~18.2 sn | ~21.4 sn |

**Yorum:**
1. **Filtre artışı:** İlk katmanda filtre sayısının 8'den 16'ya çıkması modelin yakaladığı desen çeşitliliğini artırdı ve test başarısını yaklaşık %0.9 yükseltti. Süre ise hafif uzadı.
2. **Her zaman daha iyi midir:** Hayır, filtre gereğinden fazla artırılırsa model ezberleme (overfitting) yapar ve donanımı boşuna yorar.

## 9. Bölüm: Değerlendirme ve Sonuç

Bu ödev sayesinde CNN katmanlarının görsel veriyi adım adım nasıl işlediğini, piksellerin nasıl filtrelenip boyutlarının küçüldüğünü ve Dense katmanında nasıl karara dönüştüğünü görmüş oldum. Basit 2 katmanlı bir CNN modeli bile 5 epoch gibi kısa bir sürede Fashion-MNIST veri setinde %88 civarında iyi bir başarı elde etti.